In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, when, trim

spark = SparkSession.builder \
    .appName("Aula02_ETL") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
csv_data = """id,data_venda,produto,valor,cliente_id
1,2024-01-01,Notebook,3500.00,100
2,2024-01-01,,150.50,101
3,2024-01-02,Mouse,20.00,
4,2024-01-02,Teclado,Erro,102
1,2024-01-01,Notebook,3500.00,100
5,2024-01-03,Monitor,1200.00,103"""

os.makedirs("dados", exist_ok=True)
with open("dados/vendas_sujas.csv", "w") as f:
    f.write(csv_data)

print("Arquivo 'vendas_sujas.csv' gerado!")

In [ ]:
schema_vendas = StructType([
    StructField("id", IntegerType(), True),
    StructField("data_venda", DateType(), True), # Tenta converter data
    StructField("produto", StringType(), True),
    StructField("valor", StringType(), True), # Lemos como String primeiro para tratar erros
    StructField("cliente_id", IntegerType(), True)
])

df_sujo = spark.read.csv(
    "dados/vendas_sujas.csv", 
    header=True, 
    schema=schema_vendas, # Força nosso schema
    mode="PERMISSIVE" # Permite ler linhas com erro colocando NULL
)

df_sujo.show()

In [ ]:
df_dedup = df_sujo.dropDuplicates(["id"])

df_na = df_dedup.na.fill({"produto": "Produto Desconhecido"})
df_na = df_na.na.drop(subset=["cliente_id"])

In [ ]:
df_tipos = df_na.withColumn("valor_numerico", col("valor").cast(DoubleType()))

df_final = df_tipos.na.fill({"valor_numerico": 0.0}) \
                   .drop("valor")

In [ ]:
df_final.show()
df_final.printSchema()

In [ ]:
df_final.createOrReplaceTempView("tb_vendas")

df_relatorio = spark.sql("""
    SELECT 
        produto,
        COUNT(id) as total_pedidos,
        SUM(valor_numerico) as faturamento_total,
        AVG(valor_numerico) as ticket_medio
    FROM tb_vendas
    GROUP BY produto
    ORDER BY faturamento_total DESC
""")

df_relatorio.show()

In [ ]:
caminho_parquet = "dados/relatorio_vendas.parquet"

df_relatorio.write \
    .mode("overwrite") \
    .parquet(caminho_parquet)

In [ ]:

df_leitura = spark.read.parquet(caminho_parquet)

df_leitura.show()